In [5]:
import pandas as pd

In [6]:
df = pd.read_parquet("../data/cfs_2017_cleaned.parquet",columns=['shipmt_id',
       'shipmt_value', 'shipmt_wght', 'shipmt_dist_gc', 'shipmt_dist_routed',
       'wgt_factor',
       'mode_name', 'sctg_name', 'naics_name', 'hazmat_name',
       'export_cntry_name', 'orig_state_name', 'dest_state_name',
       'orig_cfs_area_name', 'dest_cfs_area_name'])
columnName = ['mode_name','sctg_name','naics_name','hazmat_name',
       'export_cntry_name','orig_state_name','dest_state_name',
       'orig_cfs_area_name','dest_cfs_area_name']
df[columnName] = df[columnName].apply(lambda x:x.str.lower())
df.head()

,shipmt_id,shipmt_value,shipmt_wght,shipmt_dist_gc,shipmt_dist_routed,wgt_factor,mode_name,sctg_name,naics_name,hazmat_name,export_cntry_name,orig_state_name,dest_state_name,orig_cfs_area_name,dest_cfs_area_name
0,1,4380,391,54,60,328.3,company-owned truck,mixed freight,plastics and rubber products manufacturing,not hazmat,not an export,california,california,remainder of california cfs area,"fresno-madera, ca cfs area"
1,2,56,4,1524,1810,8425.3,"parcel, usps, or courier",mixed freight,electronic shopping and mail-order houses,not hazmat,not an export,utah,tennessee,"salt lake city-provo-orem, ut cfs area","knoxville-morristown-sevierville, tn cfs area"
2,3,255,440,2,5,9120.7,company-owned truck,machinery,motor vehicle and parts merchant wholesalers,not hazmat,not an export,california,california,"los angeles-long beach, ca cfs area","los angeles-long beach, ca cfs area"
3,4,250,44912,30,35,20.9,company-owned truck,natural sands,mining (except oil and gas),not hazmat,not an export,california,california,"fresno-madera, ca cfs area",remainder of california cfs area
4,5,46,73,9,11,1733.8,company-owned truck,"other coal and petroleum products, n.e.c.",fuel dealers,other hazmat,not an export,south carolina,south carolina,"greenville-spartanburg-anderson, sc cfs area","greenville-spartanburg-anderson, sc cfs area"


In [7]:
# df.info()

In [8]:
# ['shipmt_id',
#        'shipmt_value', 'shipmt_wght', 'shipmt_dist_gc', 'shipmt_dist_routed',
#        'wgt_factor',
#        'mode_name', 'sctg_name', 'naics_name', 'hazmat_name',
#        'export_cntry_name', 'orig_state_name', 'dest_state_name',
#        'orig_cfs_area_name', 'dest_cfs_area_name']

In [38]:
df["econ_val_dep"] = df["shipmt_value"] * df["wgt_factor"]
df["op_dep"] = df["shipmt_wght"] * df["wgt_factor"]

national_econ_val = df["econ_val_dep"].sum()
national_op_dep = df["op_dep"].sum()
national_econ_val

np.float64(14517252205223.7)

In [67]:
outbound_mode_dependence = df.groupby(["orig_state_name","orig_cfs_area_name","mode_name"]).agg(
       econ_val_dep = ("econ_val_dep","sum"),
       op_dep = ("op_dep","sum"),
).reset_index()
outbound_mode_dependence["econ_val_dep_total"] = outbound_mode_dependence.groupby("orig_cfs_area_name")["econ_val_dep"].transform("sum")
outbound_mode_dependence["econ_val_dep_pct"] = ((outbound_mode_dependence["econ_val_dep"]/outbound_mode_dependence["econ_val_dep_total"])*100).round(2)

outbound_mode_dependence["op_dep_total"] = outbound_mode_dependence.groupby("orig_cfs_area_name")["op_dep"].transform("sum")
outbound_mode_dependence["op_dep_pct"] = ((outbound_mode_dependence["op_dep"]/outbound_mode_dependence["op_dep_total"])*100).round(2)

# absolute value/weight actually at risk if this mode fails
outbound_mode_dependence["value_at_risk"] = (outbound_mode_dependence["econ_val_dep_pct"] / 100) * outbound_mode_dependence["econ_val_dep"]
outbound_mode_dependence["weight_at_risk"] = (outbound_mode_dependence["op_dep_pct"] / 100) * outbound_mode_dependence["op_dep"]

# how big is that risk in NATIONAL terms - does it matter beyond the region itself?
outbound_mode_dependence["value_at_risk_pct_of_national"] = (outbound_mode_dependence["value_at_risk"] / national_econ_val * 100).round(4)
outbound_mode_dependence["weight_at_risk_pct_of_national"] = (outbound_mode_dependence["weight_at_risk"] / national_op_dep * 100).round(4)

outbound_mode_dependence = outbound_mode_dependence.sort_values(by="op_dep_pct",ascending=False)

outbound_mode_dependence.head()

,orig_state_name,orig_cfs_area_name,mode_name,econ_val_dep,op_dep,econ_val_dep_total,econ_val_dep_pct,op_dep_total,op_dep_pct,value_at_risk,weight_at_risk,value_at_risk_pct_of_national,weight_at_risk_pct_of_national
1274,wyoming,remainder of wyoming cfs area,rail,5.156200e+09,5.272587e+11,2.120147e+10,24.32,6.052283e+11,87.12,1.253988e+09,4.593478e+11,0.0086,1.8420
194,district of columbia,"washington-arlington-alexandria, dc-va-md-wv c...",company-owned truck,1.173263e+09,4.443780e+09,2.296191e+09,51.10,5.461958e+09,81.36,5.995373e+08,3.615459e+09,0.0041,0.0145
270,hawaii,remainder of hawaii cfs area,company-owned truck,4.927339e+09,1.802667e+10,6.373158e+09,77.31,2.254582e+10,79.96,3.809326e+09,1.441412e+10,0.0262,0.0578
696,new hampshire,"boston-worcester-providence, ma-ri-nh-ct cfs a...",company-owned truck,1.511119e+10,4.336908e+10,4.235015e+10,35.68,5.747886e+10,75.45,5.391672e+09,3.272197e+10,0.0371,0.1312
416,kentucky,"cincinnati-wilmington-maysville, oh-ky-in cfs ...",for-hire truck,2.166454e+10,2.811895e+10,3.033878e+10,71.41,3.752929e+10,74.93,1.547065e+10,2.106953e+10,0.1066,0.0845


In [68]:
outbound_mode_dependence_skim = outbound_mode_dependence[["orig_state_name","orig_cfs_area_name","mode_name","econ_val_dep_pct","op_dep_pct","value_at_risk","weight_at_risk"]]
outbound_mode_dependence_skim.head(10)

,orig_state_name,orig_cfs_area_name,mode_name,econ_val_dep_pct,op_dep_pct,value_at_risk,weight_at_risk
1274,wyoming,remainder of wyoming cfs area,rail,24.32,87.12,1.253988e+09,4.593478e+11
194,district of columbia,"washington-arlington-alexandria, dc-va-md-wv c...",company-owned truck,51.10,81.36,5.995373e+08,3.615459e+09
270,hawaii,remainder of hawaii cfs area,company-owned truck,77.31,79.96,3.809326e+09,1.441412e+10
696,new hampshire,"boston-worcester-providence, ma-ri-nh-ct cfs a...",company-owned truck,35.68,75.45,5.391672e+09,3.272197e+10
416,kentucky,"cincinnati-wilmington-maysville, oh-ky-in cfs ...",for-hire truck,71.41,74.93,1.547065e+10,2.106953e+10
527,massachusetts,"boston-worcester-providence, ma-ri-nh-ct cfs a...",company-owned truck,28.40,71.16,1.884505e+10,1.061127e+11
805,north carolina,"raleigh-durham-chapel hill, nc cfs area",for-hire truck,42.53,70.95,1.458084e+10,5.000335e+10
124,california,"san diego-carlsbad, ca cfs area",for-hire truck,39.40,70.93,1.567975e+10,3.632575e+10
159,connecticut,"hartford-west hartford-east hartford, ct cfs area",company-owned truck,36.69,70.36,1.296791e+10,3.314079e+10
1039,tennessee,"nashville-davidson-murfreesboro, tn cfs area",for-hire truck,58.38,70.02,3.761710e+10,7.225351e+10


In [70]:
check = outbound_mode_dependence_skim.iloc[(outbound_mode_dependence_skim["econ_val_dep_pct"] >= 70) | (outbound_mode_dependence_skim["op_dep_pct"] >= 70) ]
check

,orig_state_name,orig_cfs_area_name,mode_name,econ_val_dep_pct,op_dep_pct,value_at_risk,weight_at_risk
1274,wyoming,remainder of wyoming cfs area,rail,24.32,87.12,1.253988e+09,4.593478e+11
194,district of columbia,"washington-arlington-alexandria, dc-va-md-wv c...",company-owned truck,51.10,81.36,5.995373e+08,3.615459e+09
270,hawaii,remainder of hawaii cfs area,company-owned truck,77.31,79.96,3.809326e+09,1.441412e+10
696,new hampshire,"boston-worcester-providence, ma-ri-nh-ct cfs a...",company-owned truck,35.68,75.45,5.391672e+09,3.272197e+10
416,kentucky,"cincinnati-wilmington-maysville, oh-ky-in cfs ...",for-hire truck,71.41,74.93,1.547065e+10,2.106953e+10
527,massachusetts,"boston-worcester-providence, ma-ri-nh-ct cfs a...",company-owned truck,28.40,71.16,1.884505e+10,1.061127e+11
805,north carolina,"raleigh-durham-chapel hill, nc cfs area",for-hire truck,42.53,70.95,1.458084e+10,5.000335e+10
124,california,"san diego-carlsbad, ca cfs area",for-hire truck,39.40,70.93,1.567975e+10,3.632575e+10
159,connecticut,"hartford-west hartford-east hartford, ct cfs area",company-owned truck,36.69,70.36,1.296791e+10,3.314079e+10
1039,tennessee,"nashville-davidson-murfreesboro, tn cfs area",for-hire truck,58.38,70.02,3.761710e+10,7.225351e+10


In [22]:
# check2 = outbound_mode_dependence_skim.iloc[outbound_mode_dependence_skim["orig_state_name"] == "kentucky"]
# # check2

In [24]:
# outbound_mode_dependence_skim[
#     outbound_mode_dependence_skim["orig_cfs_area_name"] == "remainder of oklahoma cfs area"
# ][["orig_state_name", "orig_cfs_area_name", "mode_name", "econ_val_dep_percentage", "op_dep_percentage"]]

In [61]:
national_scale = df.groupby(["orig_state_name","orig_cfs_area_name","mode_name"]).agg(
           area_tot_op_dep = ("op_dep","sum"),
           area_tot_econ_val_dep = ("econ_val_dep","sum")
).reset_index()
national_scale["nation_freight_pct"] = ((national_scale["area_tot_op_dep"]/national_op_dep)*100).round(2)
national_scale["nation_value_pct"] = ((national_scale["area_tot_econ_val_dep"]/national_econ_val)*100).round(2)
national_scale = national_scale.sort_values(by="nation_freight_pct",ascending=False)
national_scale.head(20)

,orig_state_name,orig_cfs_area_name,mode_name,area_tot_op_dep,area_tot_econ_val_dep,nation_freight_pct,nation_value_pct
1274,wyoming,remainder of wyoming cfs area,rail,5.272587e+11,5.156200e+09,2.11,0.04
1125,texas,remainder of texas cfs area,company-owned truck,3.835674e+11,1.176180e+11,1.54,0.81
97,california,"los angeles-long beach, ca cfs area",for-hire truck,3.546821e+11,4.324816e+11,1.42,2.98
1126,texas,remainder of texas cfs area,for-hire truck,3.409966e+11,1.244455e+11,1.37,0.86
1089,texas,"dallas-fort worth, tx-ok cfs area (tx part)",for-hire truck,3.259960e+11,2.312957e+11,1.31,1.59
1257,wisconsin,remainder of wisconsin cfs area,for-hire truck,2.993499e+11,1.339119e+11,1.20,0.92
375,iowa,remainder of iowa cfs area,for-hire truck,2.807256e+11,1.171108e+11,1.13,0.81
1108,texas,"houston-the woodlands, tx cfs area",for-hire truck,2.812963e+11,1.820576e+11,1.13,1.25
1088,texas,"dallas-fort worth, tx-ok cfs area (tx part)",company-owned truck,2.737394e+11,1.254694e+11,1.10,0.86
313,illinois,remainder of illinois cfs area,for-hire truck,2.655271e+11,1.081029e+11,1.06,0.74


In [63]:
suffering = outbound_mode_dependence_skim.merge(
    national_scale[["orig_cfs_area_name", "area_tot_op_dep", "area_tot_econ_val_dep"]],
    on="orig_cfs_area_name"
)

# absolute value/weight actually at risk if this mode fails
suffering["value_at_risk"] = (suffering["econ_val_dep_pct"] / 100) * suffering["area_tot_econ_val_dep"]
suffering["weight_at_risk"] = (suffering["op_dep_pct"] / 100) * suffering["area_tot_op_dep"]

# how big is that risk in NATIONAL terms - does it matter beyond the region itself?
suffering["value_at_risk_pct_of_national"] = (suffering["value_at_risk"] / national_econ_val * 100).round(4)
suffering["weight_at_risk_pct_of_national"] = (suffering["weight_at_risk"] / national_op_dep * 100).round(4)

suffering = suffering[suffering["op_dep_pct"] >= 70].sort_values("weight_at_risk", ascending=False)
suffering.head(20)

,orig_state_name,orig_cfs_area_name,mode_name,econ_val_dep_pct,op_dep_pct,area_tot_op_dep,area_tot_econ_val_dep,value_at_risk,weight_at_risk,value_at_risk_pct_of_national,weight_at_risk_pct_of_national
0,wyoming,remainder of wyoming cfs area,rail,24.32,87.12,5.272587e+11,5.156200e+09,1.253988e+09,4.593478e+11,0.0086,1.8420
44,massachusetts,"boston-worcester-providence, ma-ri-nh-ct cfs a...",company-owned truck,28.40,71.16,1.491184e+11,6.635581e+10,1.884505e+10,1.061127e+11,0.1298,0.4255
75,tennessee,"nashville-davidson-murfreesboro, tn cfs area",for-hire truck,58.38,70.02,1.031898e+11,6.443491e+10,3.761710e+10,7.225351e+10,0.2591,0.2897
53,north carolina,"raleigh-durham-chapel hill, nc cfs area",for-hire truck,42.53,70.95,7.047688e+10,3.428365e+10,1.458084e+10,5.000335e+10,0.1004,0.2005
62,california,"san diego-carlsbad, ca cfs area",for-hire truck,39.40,70.93,5.121351e+10,3.979633e+10,1.567975e+10,3.632575e+10,0.1080,0.1457
45,massachusetts,"boston-worcester-providence, ma-ri-nh-ct cfs a...",company-owned truck,28.40,71.16,5.021006e+10,8.255857e+10,2.344663e+10,3.572948e+10,0.1615,0.1433
69,connecticut,"hartford-west hartford-east hartford, ct cfs area",company-owned truck,36.69,70.36,4.710174e+10,3.534453e+10,1.296791e+10,3.314079e+10,0.0893,0.1329
27,new hampshire,"boston-worcester-providence, ma-ri-nh-ct cfs a...",company-owned truck,35.68,75.45,4.336908e+10,1.511119e+10,5.391672e+09,3.272197e+10,0.0371,0.1312
76,tennessee,"nashville-davidson-murfreesboro, tn cfs area",for-hire truck,58.38,70.02,4.099787e+10,1.867012e+10,1.089961e+10,2.870671e+10,0.0751,0.1151
35,kentucky,"cincinnati-wilmington-maysville, oh-ky-in cfs ...",for-hire truck,71.41,74.93,2.811895e+10,2.166454e+10,1.547065e+10,2.106953e+10,0.1066,0.0845
